In [ ]:
import zarr
import glob
import numpy as np
import matplotlib.pyplot as plt
import os
import argparse
from pathlib import Path
from scipy.optimize import curve_fit

root = Path("/Volumes/data/Sasaki/MTsingleBeads")

beads = [
        (0.63, root / "beads06um/*/*", "C0", "^", 0, 1000),
        (1.18, root/"beads1um/*/*", "C1", "o", 0, 1000),
        (3.37, root/"beads3um/*/*", "C2", "d", 0, 1000),
        (5.0, root/"beads5um/*/*", "C3", 10, 0, 1000),
        (7.24, root/"beads7um/*/*", "C4", 11, 0, 1000),
        (20.0, root/"beads20um/*/*", "C5", "s", 0, 1000),
    ]

def ring_gaussian(r, H, r_0, w):
    return - H * np.exp(- (r - r_0)**2 / (2.0 * w**2))

def get_experiment_rdfs(base_path):
    """
    指定されたパスパターンにマッチする各ディレクトリのRDFを読み込み、
    実験（ディレクトリ）ごとの平均RDFのリストを返す。
    """
    dirs = glob.glob(str(base_path))
    exp_means = []
    
    for d in dirs:
        zarr_path = os.path.join(d, "RDF.zarr")
        # RDF.zarr が存在するものだけをロード
        if os.path.exists(zarr_path):
            print(f"Loading: {zarr_path}")
            rdf = zarr.open_array(zarr_path, mode='r')[:]
            if len(rdf) > 0:
                # このディレクトリ内の全フレーム・粒子の平均を計算し、1つの実験データとする
                exp_means.append(np.nanmean(rdf, axis=0))
                
    return np.array(exp_means)

def calc_potential(base_path, scale=0.11):
    exp_means = get_experiment_rdfs(base_path)
    
    if len(exp_means) > 0:
        max_r = exp_means.shape[1]
        r = np.arange(max_r) * scale
        
        # 実験間の平均と標準誤差を計算
        rdf_mean = np.nanmean(exp_means, axis=0)
        rdf_sem = np.nanstd(exp_means, axis=0) / np.sqrt(len(exp_means))
        
        # ポテンシャルと、誤差伝播を用いたポテンシャルの標準誤差を計算
        # U(r) = -ln(g(r))
        # 誤差 dU = dg / g
        with np.errstate(divide='ignore', invalid='ignore'):
            potential_mean = -np.log(rdf_mean)
            potential_sem = rdf_sem / rdf_mean
            
        return r, rdf_mean, rdf_sem, potential_mean, potential_sem
    else:
        print(f"有効な RDF.zarr が見つかりませんでした: {base_path}")


fig_rdf, ax_rdf = plt.subplots()
fig_pot, ax_pot = plt.subplots()

for diameter, path, color, marker, min_fit, max_fit in beads:
    r, rdf_mean, rdf_sem, potential_mean, potential_sem = calc_potential(path)

    ax_rdf.plot(r, rdf_mean, label=f'{diameter} μm', color=color, marker=marker)
    ax_rdf.fill_between(r, rdf_mean - rdf_sem, rdf_mean + rdf_sem, color=color, alpha=0.3)
    ax_pot.plot(r, potential_mean, label=f'{diameter} μm', color=color, marker=marker, linestyle='None')
    ax_pot.fill_between(r, potential_mean - potential_sem, potential_mean + potential_sem, color=color, alpha=0.3)

    indices = np.where((min_fit <= r) & (r<=max_fit))    
    popt_pot, pcov_pot = curve_fit(ring_gaussian, r[indices], potential_mean[indices])

    ax_pot.plot(r[indices], ring_gaussian(r[indices], *popt_pot), color=color)

In [ ]:
fig_pot.show()